# PRECISE-DAPT Score — C-index Evaluation (Uniform vs Actual Weight)

Computes the PRECISE-DAPT score using fixed beta coefficients from the original derivation (Costa et al., *Lancet* 2017) and evaluates bleeding discrimination via C-index with 95% bootstrap CI.

**Formula:** `LP = 0.048 × age − 0.077 × Hemoglobin − 0.022 × CrCl + 0.043 × WBC + 0.55 × prior_bleeding`

**Creatinine Clearance (Cockcroft-Gault):** `CrCl = [(140 − age) × weight_kg × (0.85 if female)] / (72 × creatinine_mg_dL)`

**Three analyses on the same test set:**
1. **Uniform weight (75 kg)** — all test patients
2. **Actual weight, drop missing** — test patients with real weight only
3. **Actual weight, mean imputation** — all test patients, missing weight replaced with training set mean

**Split strategy:** Full cohort split first (random_state=44, stratified), then weight merged into test set — identical test patients across all analyses and consistent with main model evaluation.

**Notes:**
- CrCl clipped at 1 mL/min floor to handle extreme creatinine values.
- `history_of_bleeding` captures all bleeding types; PRECISE-DAPT requires prior spontaneous bleeding only — minor unavoidable limitation.
- PRECISE-DAPT predicts bleeding risk only — ischemic C-index is not applicable.

## 1. Imports & Paths

In [1]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from lifelines.utils import concordance_index

RANDOM_STATE   = 44
N_BOOTSTRAP    = 1000
ASSUMED_WEIGHT = 75.0

data_path    = "/infodev1/infodtao/AHA/Yue_UDP_data/data_table/"
lab_file     = data_path + "AHA_Baseline_characteristics_lab.csv"
bleeding_tte = data_path + "bleeding_tte_label_pw_365.csv"
weight_file  = data_path + "AHA_baseline_weight.csv"

## 2. Load Data & Merge

In [2]:
lab_df      = pd.read_csv(lab_file)
bleeding_df = pd.read_csv(bleeding_tte)
weight_df   = pd.read_csv(weight_file)[['PATIENT_CLINIC_NUMBER', 'weight_kg']]

weight_df['PATIENT_CLINIC_NUMBER'] = weight_df['PATIENT_CLINIC_NUMBER'].astype(int)
lab_df['PATIENT_CLINIC_NUMBER']    = lab_df['PATIENT_CLINIC_NUMBER'].astype(int)

df = pd.merge(
    lab_df[['PATIENT_CLINIC_NUMBER', 'age', 'Hemoglobin', 'WBC',
            'Creatinine', 'PATIENT_GENDER_NAME', 'history_of_bleeding']],
    bleeding_df,
    left_on='PATIENT_CLINIC_NUMBER', right_on='patient_id',
    how='inner'
).drop(columns=['patient_id'])

# Left join — keeps all patients, NaN for those without weight
df = pd.merge(df, weight_df, on='PATIENT_CLINIC_NUMBER', how='left')

print(f"Full cohort               : {df.shape[0]} patients")
print(f"With actual weight        : {df['weight_kg'].notna().sum()} ({df['weight_kg'].notna().mean()*100:.1f}%)")
print(f"Without weight            : {df['weight_kg'].isna().sum()}")

Full cohort               : 29032 patients
With actual weight        : 20960 (72.2%)
Without weight            : 8072


## 3. Compute Scores on Full Cohort

In [3]:
df['sex_female'] = (
    df['PATIENT_GENDER_NAME'].str.strip().str.lower() == 'female'
).astype(int)
sex_factor = np.where(df['sex_female'] == 1, 0.85, 1.0)
df['Creatinine'] = df['Creatinine'].clip(lower=0.1)

# Analysis 1: Uniform 75 kg
df['crcl_uniform'] = ((140 - df['age']) * ASSUMED_WEIGHT * sex_factor / (72 * df['Creatinine'])).clip(lower=1.0)
df['score_uniform'] = (
      0.048 * df['age']
    - 0.077 * df['Hemoglobin']
    - 0.022 * df['crcl_uniform']
    + 0.043 * df['WBC']
    + 0.55  * df['history_of_bleeding']
)

# Analysis 2 & 3: Actual weight (NaN where missing)
df['crcl_actual'] = ((140 - df['age']) * df['weight_kg'] * sex_factor / (72 * df['Creatinine'])).clip(lower=1.0)
df['score_actual'] = (
      0.048 * df['age']
    - 0.077 * df['Hemoglobin']
    - 0.022 * df['crcl_actual']
    + 0.043 * df['WBC']
    + 0.55  * df['history_of_bleeding']
)

print(f"Score (uniform) : {df['score_uniform'].notna().sum()} patients")
print(f"Score (actual)  : {df['score_actual'].notna().sum()} patients")

Score (uniform) : 28051 patients
Score (actual)  : 20616 patients


## 4. Split — Same as Main Model (random_state=44)

In [4]:
df_clean = df.dropna(subset=['score_uniform', 'time_to_event', 'bleeding_label'])

train_df, test_df = train_test_split(
    df_clean, test_size=0.2, random_state=RANDOM_STATE,
    stratify=df_clean['bleeding_label']
)
test_df = test_df.copy()

print(f"Train : {len(train_df)} patients")
print(f"Test  : {len(test_df)} patients")
print(f"Test — with actual weight : {test_df['weight_kg'].notna().sum()} ({test_df['weight_kg'].notna().mean()*100:.1f}%)")
print(f"Test — missing weight     : {test_df['weight_kg'].isna().sum()}")

Train : 22440 patients
Test  : 5611 patients
Test — with actual weight : 4130 (73.6%)
Test — missing weight     : 1481


## 5. Mean Imputation from Training Set

In [5]:
# Mean weight from training set only — no leakage
train_mean_weight = train_df['weight_kg'].mean()
print(f"Training set mean weight : {train_mean_weight:.1f} kg")

test_df['weight_imputed'] = test_df['weight_kg'].fillna(train_mean_weight)

sex_factor_test = np.where(test_df['sex_female'] == 1, 0.85, 1.0)
test_df['crcl_imputed'] = (
    (140 - test_df['age']) * test_df['weight_imputed'] * sex_factor_test
    / (72 * test_df['Creatinine'])
).clip(lower=1.0)
test_df['score_imputed'] = (
      0.048 * test_df['age']
    - 0.077 * test_df['Hemoglobin']
    - 0.022 * test_df['crcl_imputed']
    + 0.043 * test_df['WBC']
    + 0.55  * test_df['history_of_bleeding']
)

Training set mean weight : 90.3 kg


## 6. C-index with 95% Bootstrap CI — All Three Analyses

In [6]:
def bootstrap_ci(times, scores, events, n_bootstrap, random_state):
    c_index = concordance_index(times, scores, events)
    rng  = np.random.default_rng(random_state)
    n    = len(times)
    boot = []
    for _ in range(n_bootstrap):
        idx = rng.integers(0, n, n)
        try:
            boot.append(concordance_index(times[idx], scores[idx], events[idx]))
        except Exception:
            pass
    lo, hi = np.percentile(boot, [2.5, 97.5])
    return c_index, lo, hi

times  = test_df['time_to_event'].values
events = test_df['bleeding_label'].values

# Analysis 1: Uniform 75 kg — all test patients
ci1, lo1, hi1 = bootstrap_ci(times, -test_df['score_uniform'].values, events, N_BOOTSTRAP, RANDOM_STATE)

# Analysis 2: Actual weight — drop missing
mask2  = test_df['score_actual'].notna().values
ci2, lo2, hi2 = bootstrap_ci(
    times[mask2], -test_df['score_actual'].values[mask2],
    events[mask2], N_BOOTSTRAP, RANDOM_STATE
)

# Analysis 3: Actual weight — mean imputation
ci3, lo3, hi3 = bootstrap_ci(times, -test_df['score_imputed'].values, events, N_BOOTSTRAP, RANDOM_STATE)

print("=" * 65)
print(f"Analysis 1 — Uniform 75 kg, all test patients     (n={len(test_df)})")
print(f"  C-index         : {ci1:.3f}")
print(f"  95% Bootstrap CI: {lo1:.3f} - {hi1:.3f}")
print()
print(f"Analysis 2 — Actual weight, drop missing          (n={mask2.sum()})")
print(f"  C-index         : {ci2:.3f}")
print(f"  95% Bootstrap CI: {lo2:.3f} - {hi2:.3f}")
print()
print(f"Analysis 3 — Actual weight, mean imputation       (n={len(test_df)})")
print(f"  C-index         : {ci3:.3f}")
print(f"  95% Bootstrap CI: {lo3:.3f} - {hi3:.3f}")
print("=" * 65)
print()
print("Published external validation: 0.64-0.70")
print("Costa et al., Lancet 2017 | Wester et al., JAHA 2021")

Analysis 1 — Uniform 75 kg, all test patients     (n=5611)
  C-index         : 0.633
  95% Bootstrap CI: 0.614 - 0.653

Analysis 2 — Actual weight, drop missing          (n=4130)
  C-index         : 0.629
  95% Bootstrap CI: 0.607 - 0.651

Analysis 3 — Actual weight, mean imputation       (n=5611)
  C-index         : 0.629
  95% Bootstrap CI: 0.609 - 0.650

Published external validation: 0.64-0.70
Costa et al., Lancet 2017 | Wester et al., JAHA 2021


## 7. Sensitivity Analysis — Uniform Weight Values (70 / 75 / 80 kg)

In [7]:
print("Sensitivity analysis — uniform weight assumption vs C-index (same test set):")
sex_factor_test = np.where(test_df['sex_female'] == 1, 0.85, 1.0)
for w in [70, 75, 80]:
    crcl_w  = ((140 - test_df['age']) * w * sex_factor_test / (72 * test_df['Creatinine'])).clip(lower=1.0)
    score_w = (
          0.048 * test_df['age']
        - 0.077 * test_df['Hemoglobin']
        - 0.022 * crcl_w
        + 0.043 * test_df['WBC']
        + 0.55  * test_df['history_of_bleeding']
    )
    mask = score_w.notna()
    ci_w = concordance_index(
        test_df.loc[mask, 'time_to_event'].values,
        -score_w[mask].values,
        test_df.loc[mask, 'bleeding_label'].values
    )
    print(f"  Weight {w} kg -> C-index: {ci_w:.3f}")

Sensitivity analysis — uniform weight assumption vs C-index (same test set):
  Weight 70 kg -> C-index: 0.633
  Weight 75 kg -> C-index: 0.633
  Weight 80 kg -> C-index: 0.632
